In [1]:
import pandas as pd
import ast
import os
import warnings

warnings.filterwarnings("ignore")

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from Bio import SeqIO

def seq_frac_calcu(align_result, lenth, pla_acc):
    temp_id_list = list(align_result['pident'].value_counts().index)
    add_counts_a = align_result['qstart'].value_counts()
    minor_counts_a = align_result['qend'].value_counts()
    
    align_result = align_result[align_result['sseqid'].isin(pla_acc)]
    add_counts_p = align_result['qstart'].value_counts()
    minor_counts_p = align_result['qend'].value_counts()
    
    add_num_a = 0
    add_num_p = 0
    all_count_list = []
    fra_list = []
    for j in range(lenth):
        if j+1 in add_counts_a.index:
            add_num_a += add_counts_a[j+1]
        all_count = int(add_num_a)
        if j+1 in minor_counts_a.index:
            add_num_a -= minor_counts_a[j+1]
            
        if j+1 in add_counts_p.index:
            add_num_p += add_counts_p[j+1]
        pla_count = int(add_num_p)
        if j+1 in minor_counts_p.index:
            add_num_p -= minor_counts_p[j+1]
            
        try:
            fra = pla_count/all_count
        except:
            fra = 1
        fra_list.append(fra)
        all_count_list.append(all_count)
    return fra_list, all_count_list

def pla_frac_calcu(acc_n, contig, genus_name, pla_acc, dir_label):
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    acc_record = SeqIO.parse(handle, 'genbank')
    for seq_record in acc_record:
        if seq_record.id == contig:
            os.chdir('/active-data/temp/blastn')
            temp_ncl_file = open(f'temp_nucleotide_seq_{seq_record.id}.fasta', 'w+')
            SeqIO.write(seq_record, temp_ncl_file, "fasta")
            temp_ncl_file.close()
            break
    os.system(f'blastn -query temp_nucleotide_seq_{seq_record.id}.fasta -db /active-data/analysis_results/chr_pla/genus/blast_data/{genus_name}/nucleotide_seq.blastdb -out blastn_results_{seq_record.id}.txt -evalue 1e-50 -max_target_seqs 100000 -max_hsps 3000 -outfmt 6 -num_threads 8')
    head = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
    align_result = pd.read_csv(f'blastn_results_{seq_record.id}.txt', sep = '\t|;', engine = 'python', header = None, names = head)

    if '90' in dir_label:
        fra_list, all_count_list = seq_frac_calcu(align_result[align_result['pident'] >= 90], len(seq_record), pla_acc)
    elif '95' in dir_label:
        fra_list, all_count_list = seq_frac_calcu(align_result[align_result['pident'] >= 95], len(seq_record), pla_acc)
    else:
        fra_list, all_count_list = seq_frac_calcu(align_result, len(seq_record), pla_acc)
    tot_count = sum(fra_list)
    os.system(f'rm temp_nucleotide_seq_{seq_record.id}.fasta')
    os.system(f'rm blastn_results_{seq_record.id}.txt')

    return tot_count/len(seq_record), all_count_list

In [3]:
from tqdm import tqdm

for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    pla_acc = []
    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
        for item in pla_data:
            pla_acc.append(acc_n + '-' + item)

    target_folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
    contig_info = pd.read_csv(f'{target_folder}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    trans_rep = contig_info[(contig_info['category-pident_90']=='intermediate replicon')]['accession'].to_list()
    deno_data = []
    with tqdm(total = len(trans_rep), desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for replicon in trans_rep:
            acc_n, contig = replicon.split('-')
            plasmidness, all_count_list = pla_frac_calcu(acc_n, contig, genus_name, pla_acc, 'pident_90')
            temp_data = pd.DataFrame([{'contig': replicon, 'average_denominator': sum(all_count_list)/len(all_count_list), 
                                       'max_denominator': max(all_count_list), 'min_denominator': min(all_count_list)}])
            deno_data.append(temp_data)
            pbar.update(1)
    try:
        deno_data = pd.concat(deno_data, ignore_index=True)
        deno_data.to_csv(f'{target_folder}/intermediate_replicon_denominator_statistics.csv', index=False)
    except:
        pass

Klebsiella: 100%|███████████████████████████████████████████████████| 150/150 [33:08<00:00, 13.3s/B]
Staphylococcus: 100%|█████████████████████████████████████████████| 78.0/78.0 [05:40<00:00, 4.36s/B]
Pseudomonas: 100%|████████████████████████████████████████████████| 63.0/63.0 [07:26<00:00, 7.09s/B]
Salmonella: 100%|███████████████████████████████████████████████████| 109/109 [08:29<00:00, 4.67s/B]
Streptococcus: 100%|██████████████████████████████████████████████| 19.0/19.0 [00:50<00:00, 2.65s/B]
Streptomyces: 100%|███████████████████████████████████████████████| 64.0/64.0 [11:29<00:00, 10.8s/B]
Acinetobacter: 100%|██████████████████████████████████████████████| 69.0/69.0 [04:02<00:00, 3.51s/B]
Enterococcus: 100%|███████████████████████████████████████████████| 47.0/47.0 [02:28<00:00, 3.16s/B]
Bordetella: 100%|█████████████████████████████████████████████████| 2.00/2.00 [00:03<00:00, 1.82s/B]
Enterobacter: 100%|███████████████████████████████████████████████| 30.0/30.0 [01:49<00:00,